# LLM Counselor Evaluation - Memory NOT Included

This notebook evaluates LLM-generated therapeutic responses WITHOUT access to mem0 memories.

## Key Differences from Human Counselor Evaluation

- **Input**: Patient queries from transcript (NOT human counselor responses)
- **Response Source**: Generated by LLM using CBT_SYSTEM_PROMPT
- **What's Evaluated**: LLM-generated counselor responses
- **Memory Access**: NO - counselor only sees sliding window context
- **Baseline**: Fixed professional template (NOT first response)

## Features
- **Checkpointing**: Resume from any turn or transcript if interrupted
- **Markdown Logging**: All evaluations, scores, and reasoning saved to markdown
- **Output Organization**: All outputs saved to `./output_llm_counselor_memnotincluded/`

## Workflow

For each patient turn:
1. Add patient turn to mem0
2. Get conversation context (sliding window)
3. Generate LLM counselor response (NO memories)
4. Add LLM response to mem0
5. Evaluate CBT adherence
6. Evaluate persona consistency

In [ ]:
# Cell 1: Imports and Output Directory Setup
import sys
import os
import json
import time
from pathlib import Path
from datetime import datetime
from dataclasses import asdict

sys.path.append(os.path.join(os.getcwd(), "our-pipeline"))

from openai import OpenAI
from mem0 import Memory
from transcript_parser import (
    parse_html_transcript_file,
    get_conversation_context,
    get_counselor_turns,
    get_patient_turns,
    ConversationTurn
)
from mem0_integration import (
    create_mem0_config_with_llm,
    initialize_mem0,
    add_conversation_turn_to_memory,
    get_all_memories,
    audit_memories
)
from alignment_evaluators import (
    evaluate_cbt_adherence,
    evaluate_persona_consistency
)
from llm_counselor import generate_counselor_response
from therapeutic_framework import PROFESSIONAL_BASELINE_RESPONSE

# Output directory setup
OUTPUT_DIR = Path("./output_llm_counselor_memnotincluded")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)
(OUTPUT_DIR / "results").mkdir(exist_ok=True)

print("All modules loaded successfully!")
print(f"Output directory: {OUTPUT_DIR}")
print(f"  - Images: {OUTPUT_DIR}/images/")
print(f"  - Checkpoints: {OUTPUT_DIR}/checkpoints/")
print(f"  - Results: {OUTPUT_DIR}/results/")

In [28]:
# Cell 2: Configuration

# ============================================================================
# MODEL CONFIGURATION - Choose your backend
# ============================================================================

# OPTION A: Use Ollama (local, free)
USE_OLLAMA = False
OLLAMA_MODEL = "gpt-oss:20b"

# OPTION B: Use Lambda Cloud GPU instance
USE_LAMBDA_CLOUD = True
LAMBDA_CLOUD_BASE_URL = "http://localhost:11435/v1"  # Port 11435 for second Lambda instance
# OR use direct connection:
# LAMBDA_CLOUD_BASE_URL = "http://209.20.158.92:11434/v1"
LAMBDA_CLOUD_MODEL = "gpt-oss:20b"

# OPTION C: Use OpenAI API (requires API key)
USE_OPENAI = False
OPENAI_MODEL = "gpt-4o-mini"

# Separate model configuration for counselor vs judge
# (Will be set based on backend choice below)

print(f"Configuration:")
if USE_LAMBDA_CLOUD:
    COUNSELOR_MODEL = LAMBDA_CLOUD_MODEL
    JUDGE_MODEL = LAMBDA_CLOUD_MODEL
    print(f"  Backend: Lambda Cloud GPU")
    print(f"  Base URL: {LAMBDA_CLOUD_BASE_URL}")
    print(f"  Counselor Model: {COUNSELOR_MODEL}")
    print(f"  Judge Model: {JUDGE_MODEL}")
    print("  Make sure SSH tunnel is active: ssh -L 11435:localhost:11434 ubuntu@209.20.158.92")
elif USE_OLLAMA:
    COUNSELOR_MODEL = OLLAMA_MODEL
    JUDGE_MODEL = OLLAMA_MODEL
    print(f"  Backend: Ollama (local)")
    print(f"  Counselor Model: {COUNSELOR_MODEL}")
    print(f"  Judge Model: {JUDGE_MODEL}")
elif USE_OPENAI:
    COUNSELOR_MODEL = OPENAI_MODEL
    JUDGE_MODEL = OPENAI_MODEL
    print(f"  Backend: OpenAI API")
    print(f"  Counselor Model: {COUNSELOR_MODEL}")
    print(f"  Judge Model: {JUDGE_MODEL}")
else:
    raise ValueError("Please set one of USE_OLLAMA, USE_LAMBDA_CLOUD, or USE_OPENAI to True")

print(f"  Memory Access: NO")

Configuration:
  Backend: Lambda Cloud GPU
  Base URL: http://localhost:11435/v1
  Counselor Model: gpt-oss:20b
  Judge Model: gpt-oss:20b
  Make sure SSH tunnel is active: ssh -L 11435:localhost:11434 ubuntu@209.20.158.92
  Memory Access: NO


In [29]:
# Cell 3: Initialize OpenAI Client

if USE_LAMBDA_CLOUD:
    client = OpenAI(
        base_url=LAMBDA_CLOUD_BASE_URL,
        api_key="lambda"  # Ollama on Lambda doesn't need a real key
    )
    MODEL = LAMBDA_CLOUD_MODEL
    print(f"Using Lambda Cloud GPU instance")
    print(f"  Base URL: {LAMBDA_CLOUD_BASE_URL}")
    print(f"  Model: {MODEL}")
elif USE_OLLAMA:
    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
    MODEL = OLLAMA_MODEL
    print(f"Using Ollama with model: {MODEL}")
elif USE_OPENAI:
    client = OpenAI()  # Uses OPENAI_API_KEY from environment
    MODEL = OPENAI_MODEL
    print(f"Using OpenAI with model: {MODEL}")

print("\nClient created successfully!")

Using Lambda Cloud GPU instance
  Base URL: http://localhost:11435/v1
  Model: gpt-oss:20b

Client created successfully!


In [30]:
# Cell 4: Load Transcript Files from 0518-014_raw
DATASET_DIR = Path("./0518-014_raw")
transcript_files = sorted(list(DATASET_DIR.glob("*.txt")))

print(f"Found {len(transcript_files)} transcript files in {DATASET_DIR}")
print("=" * 60)

print("\nFiles to process:")
for idx, file in enumerate(transcript_files, 1):
    print(f"  {idx}. {file.name}")

Found 16 transcript files in 0518-014_raw

Files to process:
  1. 1000056544.txt
  2. 1000056545.txt
  3. 1000056546.txt
  4. 1000056547.txt
  5. 1000056548.txt
  6. 1000056549.txt
  7. 1000056550.txt
  8. 1000056551.txt
  9. 1000056552.txt
  10. 1000056553.txt
  11. 1000060755.txt
  12. 1000060756.txt
  13. 1000060757.txt
  14. 1000060758.txt
  15. 1000060759.txt
  16. 1000060760.txt


In [31]:
# Cell 5: Initialize Mem0
import shutil
import time

RESET_MEMORIES = False  # Set to True for fresh run, False to keep existing memories (use False to resume)

# Use SEPARATE ChromaDB folder for this notebook (to run in parallel with other notebooks)
CHROMA_DB_PATH = "./chroma_db_llm_counselor"

# Delete existing ChromaDB folder if reset is requested
# Handle Windows file locking issues gracefully
if RESET_MEMORIES and Path(CHROMA_DB_PATH).exists():
    try:
        shutil.rmtree(CHROMA_DB_PATH)
        print(f"Deleted existing {CHROMA_DB_PATH} folder for fresh start")
    except PermissionError as e:
        print(f"Warning: Could not delete {CHROMA_DB_PATH} - files may be locked by another process.")
        print(f"Error: {e}")
        print("To fix: Close any other notebooks/processes using this database, or restart your Python kernel.")
        print("Continuing with existing database...")
        RESET_MEMORIES = False  # Fall back to not resetting

# Unified USER_ID for persistent memory across all sessions of same patient
USER_ID = "patient_0518_014"

if USE_LAMBDA_CLOUD:
    # Get base URL without /v1 for Mem0
    mem0_base_url = LAMBDA_CLOUD_BASE_URL.replace("/v1", "")
    print(f"Mem0 base URL: {mem0_base_url}")
    
    mem_config = create_mem0_config_with_llm(
        llm_provider="ollama",  # Lambda runs Ollama
        model=LAMBDA_CLOUD_MODEL,
        base_url=mem0_base_url
    )
    mem_config["vector_store"]["config"]["collection_name"] = "llm_counselor_memnotincluded"
    mem_config["vector_store"]["config"]["path"] = CHROMA_DB_PATH  # Use separate path
    
    # Debug: print embedder config to verify URL
    print(f"Embedder config: {mem_config['embedder']}")
    
    memory = initialize_mem0(
        config=mem_config,
        reset_collection=RESET_MEMORIES
    )
    print(f"Initialized Mem0 with Lambda Cloud LLM: {LAMBDA_CLOUD_MODEL}")
elif USE_OLLAMA:
    mem_config = create_mem0_config_with_llm(
        llm_provider="ollama",
        model=OLLAMA_MODEL,
        base_url="http://localhost:11434"
    )
    mem_config["vector_store"]["config"]["collection_name"] = "llm_counselor_memnotincluded"
    mem_config["vector_store"]["config"]["path"] = CHROMA_DB_PATH  # Use separate path
    
    memory = initialize_mem0(
        config=mem_config,
        reset_collection=RESET_MEMORIES
    )
    print(f"Initialized Mem0 with Ollama LLM: {OLLAMA_MODEL}")
elif USE_OPENAI:
    memory = Memory()
    print(f"Initialized Mem0 with OpenAI")
    
if RESET_MEMORIES:
    print(f"  Reset memories for user: {USER_ID}")
print(f"Collection: llm_counselor_memnotincluded")
print(f"ChromaDB Path: {CHROMA_DB_PATH}")
print(f"USER_ID: {USER_ID} (unified across all sessions)")

Mem0 base URL: http://localhost:11435
Embedder config: {'provider': 'ollama', 'config': {'ollama_base_url': 'http://localhost:11435', 'model': 'nomic-embed-text', 'embedding_dims': 512}}
Initialized Mem0 with Lambda Cloud LLM: gpt-oss:20b
Collection: llm_counselor_memnotincluded
ChromaDB Path: ./chroma_db_llm_counselor
USER_ID: patient_0518_014 (unified across all sessions)


In [ ]:
# Cell 6: Main Processing Loop with Checkpoints and Markdown Logging

# ============================================================================
# CONFIGURATION
# ============================================================================
DELAY_BETWEEN_CALLS = 0.1 if (USE_OLLAMA or USE_LAMBDA_CLOUD) else 0.5  # Rate limiting (seconds)
RESUME_FROM_CHECKPOINT = True
VERBOSE = True  # Set to True for detailed turn-by-turn logging
baseline_response = PROFESSIONAL_BASELINE_RESPONSE

# ============================================================================
# CHECKPOINT AND MARKDOWN LOGGING FUNCTIONS
# ============================================================================

def get_checkpoint_path(transcript_file):
    return OUTPUT_DIR / "checkpoints" / f"{transcript_file.stem}_checkpoint.json"

def get_markdown_path(transcript_file):
    return OUTPUT_DIR / f"{transcript_file.stem}_evaluation_log.md"

def load_checkpoint(transcript_file):
    checkpoint_path = get_checkpoint_path(transcript_file)
    if checkpoint_path.exists() and RESUME_FROM_CHECKPOINT:
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        print(f"  Loaded checkpoint: {checkpoint['last_patient_turn_idx']} patient turns completed")
        return checkpoint
    return None

def save_checkpoint(transcript_file, checkpoint_data):
    checkpoint_path = get_checkpoint_path(transcript_file)
    with open(checkpoint_path, 'w', encoding='utf-8') as f:
        json.dump(checkpoint_data, f, indent=2, ensure_ascii=False)

def delete_checkpoint(transcript_file):
    checkpoint_path = get_checkpoint_path(transcript_file)
    if checkpoint_path.exists():
        checkpoint_path.unlink()

def init_markdown_log(transcript_file, total_turns, patient_count):
    md_path = get_markdown_path(transcript_file)
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(f"# LLM Counselor Evaluation Log: {transcript_file.name}\n\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(f"**Counselor Model:** {COUNSELOR_MODEL}\n\n")
        f.write(f"**Judge Model:** {JUDGE_MODEL}\n\n")
        f.write(f"**Memory Enhanced:** No (memories NOT passed to LLM counselor)\n\n")
        f.write(f"**Context Source:** Simulated conversation (patient + LLM responses only)\n\n")
        f.write(f"**USER_ID:** {USER_ID} (unified across all sessions)\n\n")
        f.write(f"## Transcript Info\n\n")
        f.write(f"- Total Turns: {total_turns}\n")
        f.write(f"- Patient Turns to Process: {patient_count}\n\n")
        f.write(f"---\n\n")
        f.write(f"## Turn-by-Turn Evaluations\n\n")

def append_turn_to_markdown(transcript_file, turn_number, patient_query, llm_response,
                            cbt_score, cbt_reasoning, persona_score, persona_reasoning,
                            memory_count, new_memories_this_turn):
    md_path = get_markdown_path(transcript_file)
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"### Turn {turn_number}\n\n")
        f.write(f"**Patient Query:**\n> {patient_query[:500]}{'...' if len(patient_query) > 500 else ''}\n\n")
        f.write(f"**LLM Counselor Response:**\n> {llm_response[:500]}{'...' if len(llm_response) > 500 else ''}\n\n")
        f.write(f"**CBT Adherence Score:** {cbt_score}/10\n\n")
        f.write(f"**CBT Reasoning:**\n> {cbt_reasoning}\n\n")
        f.write(f"**Persona Consistency Score:** {persona_score}/10\n\n")
        f.write(f"**Persona Reasoning:**\n> {persona_reasoning}\n\n")
        f.write(f"**Memory Stats:** (not used in generation)\n")
        f.write(f"- Total memories accumulated: {memory_count}\n")
        f.write(f"- New memories this turn: {len(new_memories_this_turn)}\n\n")
        if new_memories_this_turn:
            f.write(f"**New Memories Extracted:**\n")
            for mem in new_memories_this_turn:
                mem_text = mem.get("memory", mem.get("text", str(mem)))
                metadata = mem.get("metadata", {})
                role = metadata.get("role", "?")
                turn = metadata.get("turn_number", "?")
                f.write(f"- `[Turn {turn}, {role}]` {mem_text[:200]}{'...' if len(str(mem_text)) > 200 else ''}\n")
            f.write("\n")
        f.write(f"---\n\n")

def append_memories_to_markdown(transcript_file, memories):
    """Append complete memory dump to markdown log."""
    md_path = get_markdown_path(transcript_file)
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"## Complete Memory Dump\n\n")
        f.write(f"Total memories accumulated: {len(memories)}\n\n")
        for i, mem in enumerate(memories, 1):
            memory_text = mem.get("memory", mem.get("text", str(mem)))
            metadata = mem.get("metadata", {})
            turn = metadata.get('turn_number', '?')
            role = metadata.get('role', '?')
            f.write(f"{i}. **[Turn {turn}, {role}]** {memory_text}\n\n")

def append_summary_to_markdown(transcript_file, cbt_results, persona_results, memory_count):
    md_path = get_markdown_path(transcript_file)
    cbt_scores = [r["score"] for r in cbt_results]
    persona_scores = [r["score"] for r in persona_results]
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"## Summary Statistics\n\n")
        f.write(f"### CBT Adherence\n\n")
        f.write(f"- Mean: {sum(cbt_scores)/len(cbt_scores):.2f}/10\n")
        f.write(f"- Min: {min(cbt_scores)}/10\n")
        f.write(f"- Max: {max(cbt_scores)}/10\n\n")
        f.write(f"### Persona Consistency\n\n")
        f.write(f"- Mean: {sum(persona_scores)/len(persona_scores):.2f}/10\n")
        f.write(f"- Min: {min(persona_scores)}/10\n")
        f.write(f"- Max: {max(persona_scores)}/10\n\n")
        f.write(f"### Memory\n\n")
        f.write(f"- Total Memories Stored: {memory_count}\n\n")

def truncate(text, length=80):
    """Truncate text for display."""
    return text[:length] + "..." if len(text) > length else text

def serialize_simulated_turns(simulated_turns):
    """Convert simulated_turns to JSON-serializable format."""
    return [
        {
            "turn_number": t.turn_number,
            "role": t.role,
            "content": t.content,
            "timestamp": t.timestamp
        }
        for t in simulated_turns
    ]

def deserialize_simulated_turns(turns_data):
    """Convert JSON data back to ConversationTurn objects."""
    return [
        ConversationTurn(
            turn_number=t["turn_number"],
            role=t["role"],
            content=t["content"],
            timestamp=t.get("timestamp", "")
        )
        for t in turns_data
    ]

# ============================================================================
# MAIN PROCESSING LOOP
# ============================================================================

print(f"Starting LLM Counselor evaluation (Memory NOT Included)")
print(f"Counselor Model: {COUNSELOR_MODEL}")
print(f"Judge Model: {JUDGE_MODEL}")
print(f"Resume from checkpoint: {RESUME_FROM_CHECKPOINT}")
print(f"Verbose logging: {VERBOSE}")
print(f"Unified USER_ID: {USER_ID}")
print(f"Context Source: Simulated conversation (patient + LLM responses only)")
print(f"Total files to process: {len(transcript_files)}")
print("=" * 60)

all_file_results = []

# Track previous memories for detecting new memories per turn
previous_memory_ids = set()
initial_memories = get_all_memories(memory, USER_ID)
for mem in initial_memories:
    previous_memory_ids.add(mem.get("id", str(mem)))
print(f"Starting with {len(initial_memories)} existing memories")

for file_idx, transcript_file in enumerate(transcript_files, 1):
    print(f"\n{'=' * 60}")
    print(f"Processing file {file_idx}/{len(transcript_files)}: {transcript_file.name}")
    print(f"{'=' * 60}")
    
    print(f"Memory USER_ID: {USER_ID} (unified)")
    
    # Parse the transcript
    turns = parse_html_transcript_file(str(transcript_file))
    patient_turns = get_patient_turns(turns)
    
    print(f"Total turns: {len(turns)} ({len(patient_turns)} patient turns)")
    
    # Load checkpoint if exists
    checkpoint = load_checkpoint(transcript_file)
    
    if checkpoint:
        cbt_results = checkpoint.get('cbt_results', [])
        persona_results = checkpoint.get('persona_results', [])
        generated_responses = checkpoint.get('generated_responses', [])
        memory_snapshots = checkpoint.get('memory_snapshots', [])
        last_patient_turn_idx = checkpoint.get('last_patient_turn_idx', 0)
        # Restore simulated_turns from checkpoint
        simulated_turns_data = checkpoint.get('simulated_turns', [])
        simulated_turns = deserialize_simulated_turns(simulated_turns_data)
    else:
        cbt_results = []
        persona_results = []
        generated_responses = []
        memory_snapshots = []
        last_patient_turn_idx = 0
        simulated_turns = []  # Track patient turns + LLM responses (no human counselor)
        init_markdown_log(transcript_file, len(turns), len(patient_turns))
    
    # Get starting point for evaluation
    remaining_turns = patient_turns[last_patient_turn_idx:]
    print(f"  Patient turns to process: {len(remaining_turns)} (starting from idx {last_patient_turn_idx})")
    print(f"  Simulated turns so far: {len(simulated_turns)}")
    
    # Process remaining patient turns
    for i, patient_turn in enumerate(remaining_turns):
        current_idx = last_patient_turn_idx + i
        
        # Verbose logging: show patient turn
        if VERBOSE:
            print(f"\n  [Turn {patient_turn.turn_number}] PATIENT: {truncate(patient_turn.content, 100)}")
        else:
            print(f"\n  Processing patient turn {current_idx + 1}/{len(patient_turns)} (turn #{patient_turn.turn_number})...")
        
        # 1. Add patient turn to mem0
        add_conversation_turn_to_memory(
            memory=memory,
            turn_content=patient_turn.content,
            role="patient",
            turn_number=patient_turn.turn_number,
            user_id=USER_ID,
            verbose=VERBOSE
        )
        
        # 2. Add patient turn to simulated_turns for context building
        simulated_turns.append(ConversationTurn(
            turn_number=patient_turn.turn_number,
            role="patient",
            content=patient_turn.content,
            timestamp=""
        ))
        
        # 3. Get conversation context from SIMULATED turns (patient + LLM responses only)
        # This ensures the judge sees LLM counselor responses, not human counselor responses
        context = get_conversation_context(
            turns=simulated_turns,
            up_to_turn=patient_turn.turn_number,
            max_turns=10
        )
        
        # 4. Generate LLM counselor response (NO memories - KEY DIFFERENCE)
        llm_response = generate_counselor_response(
            client=client,
            patient_query=patient_turn.content,
            conversation_context=context,
            memories_context=None,  # NO memories for memnotincluded version
            turn_number=patient_turn.turn_number,
            model=COUNSELOR_MODEL,
            temperature=0.7
        )
        
        # Verbose logging: show generated response
        if VERBOSE:
            print(f"  [Turn {patient_turn.turn_number + 1}] LLM COUNSELOR: {truncate(llm_response, 100)}")
        
        # 5. Add LLM response to mem0
        add_conversation_turn_to_memory(
            memory=memory,
            turn_content=llm_response,
            role="counselor",
            turn_number=patient_turn.turn_number + 1,
            user_id=USER_ID,
            verbose=VERBOSE
        )
        
        # 6. Add LLM response to simulated_turns (for next iteration's context)
        simulated_turns.append(ConversationTurn(
            turn_number=patient_turn.turn_number + 1,
            role="counselor",
            content=llm_response,
            timestamp=""
        ))
        
        generated_responses.append({
            "turn_number": patient_turn.turn_number,
            "patient_query": patient_turn.content,
            "llm_response": llm_response
        })
        
        time.sleep(DELAY_BETWEEN_CALLS)
        
        # 7. Evaluate CBT adherence (context now includes LLM responses, not human)
        cbt_result = evaluate_cbt_adherence(
            client=client,
            counselor_response=llm_response,
            conversation_context=context,
            turn_number=patient_turn.turn_number,
            model=JUDGE_MODEL
        )
        cbt_results.append(asdict(cbt_result))
        
        time.sleep(DELAY_BETWEEN_CALLS)
        
        # 8. Evaluate persona consistency
        persona_result = evaluate_persona_consistency(
            client=client,
            counselor_response=llm_response,
            baseline_response=baseline_response,
            conversation_context=context,
            turn_number=patient_turn.turn_number,
            model=JUDGE_MODEL
        )
        persona_results.append(asdict(persona_result))
        
        # Get current memories and find new ones (still track even though not used in generation)
        current_memories = get_all_memories(memory, USER_ID)
        current_memory_ids = {mem.get("id", str(mem)) for mem in current_memories}
        new_memory_ids = current_memory_ids - previous_memory_ids
        new_memories_this_turn = [mem for mem in current_memories if mem.get("id", str(mem)) in new_memory_ids]
        
        # Update previous memories for next iteration
        previous_memory_ids = current_memory_ids
        
        memory_snapshots.append({
            "turn_number": patient_turn.turn_number,
            "memory_count": len(current_memories),
            "new_memories_this_turn": len(new_memories_this_turn),
            "cbt_score": cbt_result.score,
            "persona_score": persona_result.score
        })
        
        # Verbose logging: show evaluation results and memories
        if VERBOSE:
            print(f"    --> Evaluating: CBT: {cbt_result.score}/10 | Persona: {persona_result.score}/10")
            print(f"    --> Memories (not used in generation): Total: {len(current_memories)}")
            if new_memories_this_turn:
                print(f"    --> New memories extracted ({len(new_memories_this_turn)}):")
                for mem in new_memories_this_turn[:3]:  # Show first 3
                    mem_text = mem.get("memory", mem.get("text", str(mem)))
                    print(f"        + {truncate(mem_text, 70)}")
                if len(new_memories_this_turn) > 3:
                    print(f"        ... and {len(new_memories_this_turn) - 3} more")
        else:
            print(f"    CBT: {cbt_result.score}/10 | Persona: {persona_result.score}/10 | Memories: {len(current_memories)} (+{len(new_memories_this_turn)} new)")
        
        # Append to markdown log with enhanced info
        append_turn_to_markdown(
            transcript_file=transcript_file,
            turn_number=patient_turn.turn_number,
            patient_query=patient_turn.content,
            llm_response=llm_response,
            cbt_score=cbt_result.score,
            cbt_reasoning=cbt_result.reasoning,
            persona_score=persona_result.score,
            persona_reasoning=persona_result.reasoning,
            memory_count=len(current_memories),
            new_memories_this_turn=new_memories_this_turn
        )
        
        # Save checkpoint after each turn (including simulated_turns for resumption)
        checkpoint_data = {
            'filename': transcript_file.name,
            'last_patient_turn_idx': current_idx + 1,
            'total_patient_turns': len(patient_turns),
            'cbt_results': cbt_results,
            'persona_results': persona_results,
            'generated_responses': generated_responses,
            'memory_snapshots': memory_snapshots,
            'simulated_turns': serialize_simulated_turns(simulated_turns),
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        save_checkpoint(transcript_file, checkpoint_data)
        
        time.sleep(DELAY_BETWEEN_CALLS)
    
    # Get final memories and append summary to markdown
    final_memories = get_all_memories(memory, USER_ID)
    append_memories_to_markdown(transcript_file, final_memories)
    append_summary_to_markdown(transcript_file, cbt_results, persona_results, len(final_memories))
    
    # Save final results JSON
    results_path = OUTPUT_DIR / "results" / f"{transcript_file.stem}_results.json"
    results_data = {
        "filename": transcript_file.name,
        "total_turns": len(turns),
        "patient_turns_evaluated": len(cbt_results),
        "counselor_model": COUNSELOR_MODEL,
        "judge_model": JUDGE_MODEL,
        "memory_enhanced": False,
        "context_source": "simulated_conversation",
        "user_id": USER_ID,
        "generated_responses": generated_responses,
        "cbt_adherence_results": cbt_results,
        "persona_consistency_results": persona_results,
        "memory_snapshots": memory_snapshots
    }
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(results_data, f, indent=2, ensure_ascii=False)
    
    # Delete checkpoint after successful completion
    delete_checkpoint(transcript_file)
    
    all_file_results.append(results_data)
    print(f"\n  Completed: {transcript_file.name}")
    print(f"  Results: {results_path}")
    print(f"  Markdown log: {get_markdown_path(transcript_file)}")

print(f"\n{'=' * 60}")
print(f"ALL FILES PROCESSED!")
print(f"Total memories accumulated: {len(get_all_memories(memory, USER_ID))}")
print(f"Output directory: {OUTPUT_DIR}/")
print(f"{'=' * 60}")

In [ ]:
# Cell 7: Memory Audit
print("\nPerforming memory audit...")
all_memories = get_all_memories(memory, USER_ID)
print(f"Total memories stored: {len(all_memories)}")

audit_result = audit_memories(
    client=client,
    memories=all_memories,
    model=JUDGE_MODEL
)

print(f"\nMemory Audit Results:")
print(f"  Total Memories: {audit_result.total_memories}")
print(f"  Distortion Count: {audit_result.distortion_count}")
print(f"  Collusion Score: {audit_result.collusion_score:.2f}")
print(f"  Reasoning: {audit_result.reasoning}")

In [ ]:
# Cell 8: Save Results
output = {
    "metadata": {
        "transcript_source": "SAMPLE_TRANSCRIPT",
        "total_patient_turns": len(patient_turns),
        "counselor_model": COUNSELOR_MODEL,
        "judge_model": JUDGE_MODEL,
        "memories_passed_to_counselor": False,
        "memories_passed_to_judge": False,
        "baseline_type": "fixed_professional_template"
    },
    "generated_responses": generated_responses,
    "cbt_results": cbt_results,
    "persona_results": persona_results,
    "memory_audit": asdict(audit_result),
    "statistics": {
        "avg_cbt_score": sum(r["score"] for r in cbt_results) / len(cbt_results) if cbt_results else 0,
        "avg_persona_score": sum(r["score"] for r in persona_results) / len(persona_results) if persona_results else 0,
        "collusion_score": audit_result.collusion_score
    }
}

output_file = "llm_counselor_memnotincluded.json"
with open(output_file, "w") as f:
    json.dump(output, f, indent=2)

print(f"\nResults saved to {output_file}")
print(f"\nSummary Statistics:")
print(f"  Average CBT Score: {output['statistics']['avg_cbt_score']:.2f}/10")
print(f"  Average Persona Score: {output['statistics']['avg_persona_score']:.2f}/10")
print(f"  Memory Collusion Score: {output['statistics']['collusion_score']:.2f}")

In [ ]:
# Cell 9: Visualization
import matplotlib.pyplot as plt
import numpy as np

if all_file_results:
    # Aggregate all results
    all_cbt_scores = []
    all_persona_scores = []
    all_memory_counts = []
    
    for result in all_file_results:
        all_cbt_scores.extend([r["score"] for r in result["cbt_adherence_results"]])
        all_persona_scores.extend([r["score"] for r in result["persona_consistency_results"]])
        all_memory_counts.extend([s["memory_count"] for s in result["memory_snapshots"]])
    
    eval_numbers = list(range(1, len(all_cbt_scores) + 1))
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    
    # Plot 1: CBT Adherence
    axes[0].plot(eval_numbers, all_cbt_scores, 'b-', alpha=0.7)
    axes[0].axhline(y=7, color='green', linestyle='--', label='Target (7+)')
    window = min(20, len(all_cbt_scores)//5) if len(all_cbt_scores) > 20 else 5
    rolling_avg = np.convolve(all_cbt_scores, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window//2, len(rolling_avg) + window//2), rolling_avg, 'b-', linewidth=2, label=f'Rolling Avg ({window})')
    axes[0].set_ylabel('CBT Score')
    axes[0].set_title(f'LLM Counselor CBT Adherence Over Time (Memory NOT Included)\nUnified USER_ID: {USER_ID}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(0, 11)
    
    # Plot 2: Persona Consistency
    axes[1].plot(eval_numbers, all_persona_scores, 'g-', alpha=0.7)
    axes[1].axhline(y=7, color='green', linestyle='--', label='Target (7+)')
    rolling_avg2 = np.convolve(all_persona_scores, np.ones(window)/window, mode='valid')
    axes[1].plot(range(window//2, len(rolling_avg2) + window//2), rolling_avg2, 'g-', linewidth=2, label=f'Rolling Avg ({window})')
    axes[1].set_ylabel('Persona Score')
    axes[1].set_title('LLM Counselor Persona Consistency Over Time (Memory NOT Included)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, 11)
    
    # Plot 3: Memory Growth - Unified across all sessions (tracked but not used)
    axes[2].plot(eval_numbers, all_memory_counts, 'm-', linewidth=1.5, label='Cumulative Memories (Single Patient)')
    axes[2].fill_between(eval_numbers, 0, all_memory_counts, alpha=0.2, color='purple')
    axes[2].set_xlabel('Cumulative Evaluations (all 16 sessions)')
    axes[2].set_ylabel('Memory Count')
    axes[2].set_title(f'Unified Memory Accumulation - Single Patient Across All Sessions\nUSER_ID: {USER_ID} (memories tracked, not used in generation)')
    axes[2].legend(loc='upper left')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    image_path = OUTPUT_DIR / "images" / "llm_counselor_overview.png"
    plt.savefig(image_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nFigure saved to {image_path}")
    print(f"Total evaluations: {len(all_cbt_scores)}")
    print(f"Final memory count: {all_memory_counts[-1] if all_memory_counts else 0}")
    
    # Per-file visualizations
    for result in all_file_results:
        filename = result["filename"].replace(".txt", "")
        cbt_scores = [r["score"] for r in result["cbt_adherence_results"]]
        persona_scores = [r["score"] for r in result["persona_consistency_results"]]
        memory_counts = [s["memory_count"] for s in result["memory_snapshots"]]
        turns = [r["turn_number"] for r in result["cbt_adherence_results"]]
        
        fig, axes = plt.subplots(3, 1, figsize=(12, 10))
        
        axes[0].plot(turns, cbt_scores, 'b-o', markersize=3)
        axes[0].axhline(y=7, color='green', linestyle='--')
        axes[0].set_ylabel('CBT Score')
        axes[0].set_title(f'{filename} - LLM Counselor CBT (No Memory)')
        axes[0].set_ylim(0, 11)
        axes[0].grid(True, alpha=0.3)
        
        axes[1].plot(turns, persona_scores, 'g-o', markersize=3)
        axes[1].axhline(y=7, color='green', linestyle='--')
        axes[1].set_ylabel('Persona Score')
        axes[1].set_title(f'{filename} - LLM Counselor Persona (No Memory)')
        axes[1].set_ylim(0, 11)
        axes[1].grid(True, alpha=0.3)
        
        axes[2].plot(turns, memory_counts, 'm-s', markersize=3)
        axes[2].fill_between(turns, 0, memory_counts, alpha=0.2, color='purple')
        axes[2].set_xlabel('Turn Number')
        axes[2].set_ylabel('Memory Count')
        axes[2].set_title(f'{filename} - Memory Growth (Unified Patient: {USER_ID})')
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        file_image_path = OUTPUT_DIR / "images" / f"{filename}_evaluation.png"
        plt.savefig(file_image_path, dpi=150, bbox_inches='tight')
        plt.close()
    
    print(f"Per-file images saved to {OUTPUT_DIR / 'images'}/")